<a href="https://colab.research.google.com/github/gautamkr1876/AIML_ClassNotes/blob/main/8.%20Agentic%20AI%20Systems/12.%20%20Hybrid%20Search%20%26%20Advanced%20Retrieval/L11_and_L12_Hybrid_Search_%26_Advanced_Retrieval.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Hybrid Search and Advanced Retrieval

Last class we built an agent that planned retrieval, routed subquestions to SQL, vector, and graph tools, and looped until it had enough evidence. Today we go one level deeper. Instead of trusting whatever the vector tool returns, we are going to make the vector tool itself smart. By the end of this session, our retriever will combine keyword and semantic search, rerank its own shortlist, judge whether the results are actually useful, and retry with a better query if they are not.

**Think of it this way:** last class the agent was the manager. Today we upgrade the employee.

## Section 0: Setup and Recap

Quick reconnect to last class. In the chat, drop one line: what did the vector tool actually do in our agent loop last time? I will wait 30 seconds and then we move on.

The short answer is: it did a single top-k cosine similarity lookup. That is fine for easy questions. It falls apart the moment a user types an exact identifier, an acronym, or a query where the corpus uses different vocabulary than the user. That is what we are fixing today.

Let us get the environment up. If you ran the previous notebook end-to-end, most of this is already installed.

In [ ]:
# Install the pieces we need. sentence-transformers and groq are almost
# certainly already present from the previous class. rank-bm25 is new.
!pip -q install sentence-transformers groq rank-bm25 numpy pandas
print("Dependencies ready.")

Dependencies ready.


In [ ]:
# Standard imports and credential entry. Same pattern as last class:
# environment first, prompt only if missing. Groq is optional; without it,
# a couple of demos will use a small deterministic fallback.
import os
import json
import re
import math
import numpy as np
import pandas as pd


# Attempt to load GROQ_API_KEY from Colab secrets if not already in environment
if not os.environ.get("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        groq_key = userdata.get('GROQ_API_KEY')
        if groq_key:
            os.environ["GROQ_API_KEY"] = groq_key
            print("Groq API key loaded from Colab secrets.")
        else:
            print("GROQ_API_KEY not found in Colab secrets.")
    except ImportError:
        print("Running outside Colab or google.colab.userdata not available.")
    except Exception as e:
        print(f"Error loading Groq API key from Colab secrets: {e}")

if not os.environ.get("GROQ_API_KEY"):
    print("Please set your Groq API key in Colab Secrets (Tools -> Secrets) as GROQ_API_KEY.")
    print("Skipping Groq client initialization.")
else:
    print("Groq API key found in environment.")

print("Groq key present:", bool(os.environ.get("GROQ_API_KEY")))

Groq API key found in environment.
Groq key present: True


In [ ]:
# Load the embedding model we used last class. Cached after first use.
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_texts(texts):
    vecs = EMBED_MODEL.encode(list(texts), normalize_embeddings=True)
    return np.asarray(vecs, dtype="float32")

print("Embedding model ready. Dimension:", EMBED_MODEL.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model ready. Dimension: 384


/tmp/ipykernel_3332/4165607517.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding model ready. Dimension:", EMBED_MODEL.get_sentence_embedding_dimension())


In [ ]:
# Groq wrapper. Same shape as last class, minimal offline fallback.
def _offline_fallback(user):
    text = (user or "").lower()
    if "reformulate" in text or "rewrite" in text or "paraphrase" in text:
        return json.dumps({"queries": [
            "LLM prompt injection defenses",
            "guardrails against jailbreak attacks in language models",
            "input sanitization for model endpoints"
        ]})
    if "verdict" in text:
        return json.dumps({"verdict": "insufficient", "reason": "evidence is off-topic"})
    return "Offline fallback: no answer generated."

class LLM:
    def __init__(self, model="openai/gpt-oss-120b"):
        self.model = model
        self.client = None
        if os.environ.get("GROQ_API_KEY"):
            try:
                from groq import Groq
                self.client = Groq(api_key=os.environ["GROQ_API_KEY"])
            except Exception as exc:
                print("Groq init failed, using offline fallback:", exc)

    def complete(self, system, user, temperature=0.0):
        if self.client is None:
            return _offline_fallback(user)
        resp = self.client.chat.completions.create(
            model=self.model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )
        return resp.choices[0].message.content

def parse_json(text):
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if "\n" in cleaned:
            first, rest = cleaned.split("\n", 1)
            if first.strip().lower() in ("json", ""):
                cleaned = rest
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return json.loads(cleaned)

llm = LLM()
print("LLM mode:", "live" if llm.client else "offline")

LLM mode: live


### Extending the corpus

Last class we worked with 6 documents about the same fictional company. That was enough to show the concepts, but it is far too small for hybrid search to look interesting. With 6 documents, every retriever returns roughly the same top-3. Today we extend it to 20 richer documents on the same company: postmortems, RFCs, incident reports, policies, project deep-dives. Same fictional org, more depth.

Chat prompt while I run the next cell: which document type do you think will trip up pure semantic search the most, and why?

In [ ]:
# Extended corpus for today. Same fictional company as last class, but 20 documents
# spanning several genres. Deliberately includes exact identifiers (INC-2847,
# RFC-014) and acronyms, which is where BM25 will earn its keep.
DOCUMENTS = [
    ("D01", "project", "Guardrail Overview",
     "Project Guardrail hardens the company LLM endpoints against prompt injection and "
     "jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming."),

    ("D02", "project", "Sentinel Overview",
     "Sentinel provides continuous LLM security monitoring of production model traffic, "
     "detecting adversarial prompts and potential data exfiltration in real time."),

    ("D03", "project", "Atlas Overview",
     "Atlas is the unified data platform providing feature storage, batch pipelines, and "
     "governed access to analytics tables across the company."),

    ("D04", "project", "Nimbus Overview",
     "Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh "
     "that host internal applications."),

    ("D05", "project", "Prism Overview",
     "Prism is the shared design system defining components, typography, and accessibility "
     "guidelines used across product surfaces."),

    ("D06", "incident", "INC-2847 Postmortem",
     "Incident INC-2847 involved a prompt injection attack against the Guardrail-protected "
     "chat endpoint. An attacker embedded instructions in a support ticket that caused the model "
     "to reveal internal routing tokens. Root cause: output filter regression after the v2.3 deploy."),

    ("D07", "incident", "INC-3102 Postmortem",
     "Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused "
     "cascading pod evictions during peak traffic. Root cause: an HPA threshold set below the "
     "pod startup time."),

    ("D08", "incident", "INC-2915 Postmortem",
     "Incident INC-2915 was a data pipeline stall in Atlas after a schema evolution in the "
     "orders table. Downstream feature jobs failed silently for six hours before alerting fired."),

    ("D09", "rfc", "RFC-014 Cross-Encoder Reranking",
     "This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the "
     "internal assistant. Expected latency impact: 40 to 90 milliseconds per query. Recommended "
     "shortlist size: 20."),

    ("D10", "rfc", "RFC-021 Query Expansion Pipeline",
     "This RFC introduces LLM-based query expansion for the internal assistant. Rationale: "
     "vocabulary mismatch between user phrasing and technical documents. Proposes three "
     "paraphrases per query fused via reciprocal rank fusion."),
    ("D11", "rfc", "RFC-008 Read-Only SQL Guardrails",
     "Establishes token-based SQL validation, read-only connections, and row limits for "
     "agent-authored queries against the org database."),
    ("D12", "policy", "Model Input Security Policy",
     "Company policy requires every project that handles model inputs to complete an LLM "
     "security threat model and to document mitigations for prompt injection before launch."),
    ("D13", "policy", "Data Retention Policy",
     "Personal data is retained for at most 90 days in operational stores. Aggregated analytics "
     "data may be retained for 24 months. Legal holds override deletion."),
    ("D14", "policy", "On-Call Rotation Policy",
     "Engineers on the primary on-call rotation are compensated for shifts outside business "
     "hours. Handoff occurs at 09:00 local time with a documented status transfer."),
    ("D15", "deepdive", "How Guardrail Handles Jailbreaks",
     "The Guardrail defense pipeline layers several techniques against jailbreak attempts: "
     "instruction hierarchy tagging, adversarial suffix detection, and periodic red-team "
     "evaluations against a curated attack set."),
    ("D16", "deepdive", "Atlas Feature Store Internals",
     "Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a "
     "snapshot table to prevent training-serving skew. The batch layer runs nightly on Spark."),
    ("D17", "deepdive", "Nimbus Service Mesh Design",
     "Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Retry budgets are set "
     "per service; circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds."),
    ("D18", "onboarding", "New Hire Onboarding Guide",
     "New engineers are paired with a buddy for the first two weeks, complete the security "
     "training module, and ship a small starter task to production by end of week two."),
    ("D19", "onboarding", "Access Request Procedure",
     "To request access to production systems, file a ticket in the access portal, cite the "
     "business justification, and obtain approval from the owning team's manager."),
    ("D20", "deepdive", "Sentinel Detection Rules",
     "Sentinel evaluates model outputs against a set of detection rules: PII patterns, "
     "secret formats, and known adversarial suffixes. Alerts route to the security on-call."),
]

df = pd.DataFrame(DOCUMENTS, columns=["id", "type", "title", "text"])
print("Total documents:", len(DOCUMENTS))
print("By type:")
print(df["type"].value_counts().to_string())

Total documents: 20
By type:
type
project       5
deepdive      4
incident      3
rfc           3
policy        3
onboarding    2


In [ ]:
df

,id,type,title,text
0,D01,project,Guardrail Overview,Project Guardrail hardens the company LLM endp...
1,D02,project,Sentinel Overview,Sentinel provides continuous LLM security moni...
2,D03,project,Atlas Overview,Atlas is the unified data platform providing f...
3,D04,project,Nimbus Overview,Nimbus manages the Kubernetes based infrastruc...
4,D05,project,Prism Overview,Prism is the shared design system defining com...
5,D06,incident,INC-2847 Postmortem,Incident INC-2847 involved a prompt injection ...
6,D07,incident,INC-3102 Postmortem,Incident INC-3102 was a Kubernetes autoscaler ...
7,D08,incident,INC-2915 Postmortem,Incident INC-2915 was a data pipeline stall in...
8,D09,rfc,RFC-014 Cross-Encoder Reranking,This RFC proposes adopting cross-encoder reran...
9,D10,rfc,RFC-021 Query Expansion Pipeline,This RFC introduces LLM-based query expansion ...


In [ ]:
# Precompute embeddings for all 20 documents. We will reuse this matrix everywhere.
DOC_TEXTS = ["{}. {}".format(d[2], d[3]) for d in DOCUMENTS]
DOC_IDS = [d[0] for d in DOCUMENTS]
DOC_TITLES = [d[2] for d in DOCUMENTS]

DOC_MATRIX = embed_texts(DOC_TEXTS)
print("Embedding matrix shape:", DOC_MATRIX.shape)

Embedding matrix shape: (20, 384)


In [ ]:
DOC_MATRIX[0]

array([-4.30591777e-02,  5.08917831e-02,  3.57589610e-02, -1.62100941e-02,
        7.90967196e-02,  2.88818050e-02,  7.92796537e-02,  5.09099737e-02,
       -5.44830486e-02,  1.29518181e-03,  1.82403717e-02,  4.81646918e-02,
        3.76657508e-02,  2.62153167e-02,  2.90895626e-02,  1.03270449e-01,
        9.78823230e-02,  4.10558619e-02,  3.12298276e-02, -1.75279547e-02,
        5.15139215e-02, -7.15001347e-03, -1.76462643e-02,  6.03828318e-02,
       -9.28745791e-02,  2.73915194e-02,  1.55219890e-03,  5.47848493e-02,
       -4.36043926e-02, -8.77165869e-02, -7.65803382e-02, -6.95285127e-02,
       -5.49631715e-02,  4.01412025e-02,  3.25146317e-03, -8.17873806e-04,
        1.89566035e-02,  1.39995692e-02, -5.80907129e-02, -1.92554928e-02,
       -4.50294316e-02,  5.52520528e-02,  2.17561033e-02,  3.36357392e-02,
        2.92716287e-02, -9.89933964e-03,  1.91215537e-02, -4.27691787e-02,
       -8.49268958e-02, -5.24811894e-02,  6.38536885e-02,  2.24280618e-02,
        4.21925224e-02, -

## Section 1: BM25 + Semantic Hybrid Search

Here is the failure mode we are trying to fix. When a user searches for the postmortem of incident INC-2847, they type that identifier verbatim. Dense embeddings squish exact strings into fuzzy meaning. The word INC-2847 sits in embedding space near other incident-shaped things, not necessarily near the specific document that contains that specific ID.

The analogy is Google Search. When you type an exact error code, Google is not doing pure semantic matching. It combines keyword hits with semantic understanding, and the exact keyword match usually wins for identifiers. That combination is what we are building now.

Let us see dense retrieval fail on the identifier query.

## BM25 Example

D1: "machine learning is powerful"

D2: "machine learning machine learning"

D3: "deep learning is useful"

Q = "machine learning"

N = 3 (No of documents)

| Document | Text                              | Length |
| -------- | --------------------------------- | -----: |
| D1       | machine learning is powerful      |      4 |
| D2       | machine learning machine learning |      4 |
| D3       | deep learning is useful           |      4 |


Calculate IDF


calculate for the term machine

Term: "machine"

How many documents contain "machine"?

D1 → yes
D2 → yes
D3 → no


df(machine) = 2



In [ ]:
# Pure dense retrieval on an identifier-heavy query.
def dense_search(query, k=5):
    q = embed_texts([query])[0]
    scores = DOC_MATRIX @ q
    order = np.argsort(-scores)[:k]
    return [(DOC_IDS[i], DOC_TITLES[i], float(scores[i])) for i in order]

query = "INC-2847 postmortem"
print("Query:", query)
print()
print("Pure dense top-5:")
for doc_id, title, score in dense_search(query, k=5):
    print("  {:.3f}  {}  ({})".format(score, title, doc_id))

Query: INC-2847 postmortem

Pure dense top-5:
  0.503  INC-2847 Postmortem  (D06)
  0.466  INC-2915 Postmortem  (D08)
  0.402  INC-3102 Postmortem  (D07)
  0.150  Access Request Procedure  (D19)
  0.108  RFC-014 Cross-Encoder Reranking  (D09)


Notice how dense retrieval ranks generic incident and security docs above the actual INC-2847 postmortem, or ties it with irrelevant ones. The embedding model does not treat the string INC-2847 as special. Now let us build BM25, which does.

In [ ]:
# BM25 using the rank-bm25 library. Same tokenization for docs and queries.
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"[a-z0-9\-]+", text.lower())

TOKENIZED_DOCS = [tokenize(t) for t in DOC_TEXTS]
BM25 = BM25Okapi(TOKENIZED_DOCS)

def bm25_search(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    order = np.argsort(-scores)[:k]
    return [(DOC_IDS[i], DOC_TITLES[i], float(scores[i])) for i in order]

print("BM25 top-5 for the same query:")
for doc_id, title, score in bm25_search(query, k=5):
    print("  {:.3f}  {}  ({})".format(score, title, doc_id))

BM25 top-5 for the same query:
  4.501  INC-2847 Postmortem  (D06)
  1.553  INC-3102 Postmortem  (D07)
  1.553  INC-2915 Postmortem  (D08)
  0.000  Guardrail Overview  (D01)
  0.000  Nimbus Overview  (D04)


BM25 nails the exact identifier because it treats INC-2847 as a rare token, gives it a large IDF, and any document containing it shoots to the top. But BM25 is not smart about meaning. Watch what happens when the user rephrases the same intent with different vocabulary.

In [ ]:
# BM25 struggles when vocabulary shifts.
paraphrase = "how do we stop attackers from tricking our chatbot with clever prompts"

print("Query:", paraphrase)
print()
print("BM25 top-5:")
for doc_id, title, score in bm25_search(paraphrase, k=5):
    print("  {:.3f}  {}  ({})".format(score, title, doc_id))
print()
print("Dense top-5:")
for doc_id, title, score in dense_search(paraphrase, k=5):
    print("  {:.3f}  {}  ({})".format(score, title, doc_id))

Query: how do we stop attackers from tricking our chatbot with clever prompts

BM25 top-5:
  2.829  Sentinel Overview  (D02)
  2.514  How Guardrail Handles Jailbreaks  (D15)
  2.514  Access Request Procedure  (D19)
  1.993  On-Call Rotation Policy  (D14)
  1.817  New Hire Onboarding Guide  (D18)

Dense top-5:
  0.311  INC-2847 Postmortem  (D06)
  0.251  How Guardrail Handles Jailbreaks  (D15)
  0.248  Sentinel Overview  (D02)
  0.226  Sentinel Detection Rules  (D20)
  0.210  Model Input Security Policy  (D12)


The paraphrase does not share the technical vocabulary of the corpus (prompt injection, jailbreak). BM25 scores drop toward zero or match irrelevant docs by chance. Dense embeddings, which encode meaning, correctly surface Guardrail and Sentinel.

So the two retrievers fail in complementary ways. We fuse them.

The Netflix analogy: when Netflix ranks candidates for your home page, one signal says "people similar to you watched this" and another says "this genre matches your history". These two signals are on completely different scales. If Netflix tried to average them numerically it would be a disaster. Instead the industry standard trick is Reciprocal Rank Fusion, which uses only the position of each item in each ranking, not the raw score. Position is universal. Scores are not.

In [ ]:
# Reciprocal Rank Fusion. Combines ranked lists by position, scale-invariant.
def reciprocal_rank_fusion(rankings, k=60):
    fused = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(fused.items(), key=lambda p: -p[1])

def hybrid_search(query, k=5, shortlist=15):
    bm25_ids = [doc_id for doc_id, _, _ in bm25_search(query, k=shortlist)]
    dense_ids = [doc_id for doc_id, _, _ in dense_search(query, k=shortlist)]
    fused = reciprocal_rank_fusion([bm25_ids, dense_ids])[:k]
    id_to_title = dict(zip(DOC_IDS, DOC_TITLES))
    return [(doc_id, id_to_title[doc_id], score) for doc_id, score in fused]

for q in ["INC-2847 postmortem", paraphrase]:
    print("Query:", q)
    print("Hybrid top-5:")
    for doc_id, title, score in hybrid_search(q, k=5):
        print("  {:.4f}  {}  ({})".format(score, title, doc_id))
    print()

Query: INC-2847 postmortem
Hybrid top-5:
  0.0328  INC-2847 Postmortem  (D06)
  0.0320  INC-3102 Postmortem  (D07)
  0.0320  INC-2915 Postmortem  (D08)
  0.0299  Guardrail Overview  (D01)
  0.0299  RFC-014 Cross-Encoder Reranking  (D09)

Query: how do we stop attackers from tricking our chatbot with clever prompts
Hybrid top-5:
  0.0323  Sentinel Overview  (D02)
  0.0323  How Guardrail Handles Jailbreaks  (D15)
  0.0315  INC-2847 Postmortem  (D06)
  0.0304  Access Request Procedure  (D19)
  0.0299  Guardrail Overview  (D01)



Hybrid wins both cases. It surfaces INC-2847 for the exact identifier and Guardrail-related docs for the paraphrase. Neither retriever alone gets both right.

The mathematical form of RRF is worth internalizing: $\text{RRF}(d) = \sum_{r \in \text{rankings}} \frac{1}{k + \text{rank}_r(d)}$ where $k$ is a smoothing constant, usually 60. Notice the equation contains no reference to the underlying scores. Only ranks.

| Document | BM25 Rank | Vector Rank |         RRF |
| -------- | --------: | ----------: | ----------: |
| **D1**   |         1 |           2 | **0.03252** |
| **D2**   |         3 |           1 | **0.03226** |
| **D3**   |         2 |           4 | **0.03176** |
| D4       |         — |           3 |     0.01587 |


### MCQ 1

**Why does Reciprocal Rank Fusion combine BM25 and dense retrieval more robustly than a weighted sum of their scores?**

**A.** RRF uses machine learning to learn the optimal weights per query, while a weighted sum is fixed.

**B.** RRF depends only on the rank position of each document, so it is invariant to the incomparable score scales of BM25 and cosine similarity.

**C.** RRF filters out low-scoring documents before combining, which a weighted sum cannot do.

**D.** RRF is faster because it avoids computing scores at all.

**Answer: B.** BM25 scores are unbounded and query-dependent, while cosine similarity is in $[-1, 1]$. A weighted sum requires normalizing these onto a common scale, which is fragile and query-dependent. RRF sidesteps normalization entirely by using only rank positions, which are on the same scale by definition.

## Section 2: Cross-Encoder Reranking

Hybrid retrieval gives us a strong shortlist. But shortlists are not final answers. The next step is reranking, where a heavier model reads the query and each candidate document together and scores their relevance jointly.

The bi-encoder versus cross-encoder distinction matters. A bi-encoder, which is what our dense retriever is, computes an embedding for the query and an embedding for each document independently, then compares them by cosine. That is cheap and scalable. A cross-encoder feeds the query and document into the model as one input and outputs a single relevance score. It is slower but far more accurate because the model can attend to interactions between the query and the document.

The hiring analogy: bi-encoder retrieval is like a resume screener that ranks resumes based on keyword and skill embeddings, without looking at the job description in detail. Fast, coarse, useful for narrowing 10000 resumes to 50. A cross-encoder is the actual interview, where the interviewer reads the resume alongside the specific job description and considers how well they match. Slow, precise, useless as a first-stage filter.

Nobody, and I mean nobody, runs a cross-encoder against a full corpus. The Airbnb-style pattern is: cheap retrieval narrows 500000 listings to 300 candidates, then a heavier ranker considers your specific dates, party size, and query text to reorder the top 50.

In [ ]:
# Load a real cross-encoder. Small, MS MARCO trained, runs on CPU.
# First-time download is around 90 MB.
from sentence_transformers import CrossEncoder

RERANKER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder ready.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder ready.


In [ ]:
# Rerank a hybrid shortlist with the cross-encoder.
def rerank(query, doc_ids, top_k=5):
    id_to_text = dict(zip(DOC_IDS, DOC_TEXTS))
    id_to_title = dict(zip(DOC_IDS, DOC_TITLES))
    pairs = [(query, id_to_text[d]) for d in doc_ids]
    scores = RERANKER.predict(pairs)
    reranked = sorted(zip(doc_ids, scores), key=lambda p: -p[1])[:top_k]
    return [(d, id_to_title[d], float(s)) for d, s in reranked]

def hybrid_then_rerank(query, shortlist=10, top_k=5):
    shortlist_ids = [d for d, _, _ in hybrid_search(query, k=shortlist)]
    return shortlist_ids, rerank(query, shortlist_ids, top_k=top_k)

for q in [
    "how do we protect the chatbot from adversarial prompts",
    "what happened during the Kubernetes autoscaling outage",
]:
    print("Query:", q)
    shortlist_ids, reranked = hybrid_then_rerank(q, shortlist=10, top_k=5)
    id_to_title = dict(zip(DOC_IDS, DOC_TITLES))
    print("Hybrid shortlist (order before reranking):")
    for i, d in enumerate(shortlist_ids[:5], 1):
        print("  {}. {} ({})".format(i, id_to_title[d], d))
    print("After cross-encoder reranking:")
    for i, (d, title, score) in enumerate(reranked, 1):
        print("  {}. {:+.3f}  {} ({})".format(i, score, title, d))
    print()

Query: how do we protect the chatbot from adversarial prompts
Hybrid shortlist (order before reranking):
  1. Sentinel Overview (D02)
  2. How Guardrail Handles Jailbreaks (D15)
  3. Sentinel Detection Rules (D20)
  4. INC-2847 Postmortem (D06)
  5. Access Request Procedure (D19)
After cross-encoder reranking:
  1. -2.572  INC-2847 Postmortem (D06)
  2. -3.468  Sentinel Overview (D02)
  3. -5.773  How Guardrail Handles Jailbreaks (D15)
  4. -6.757  Sentinel Detection Rules (D20)
  5. -9.605  Guardrail Overview (D01)

Query: what happened during the Kubernetes autoscaling outage
Hybrid shortlist (order before reranking):
  1. Nimbus Overview (D04)
  2. INC-3102 Postmortem (D07)
  3. INC-2847 Postmortem (D06)
  4. New Hire Onboarding Guide (D18)
  5. INC-2915 Postmortem (D08)
After cross-encoder reranking:
  1. +5.250  INC-3102 Postmortem (D07)
  2. -2.251  Nimbus Overview (D04)
  3. -10.925  INC-2847 Postmortem (D06)
  4. -11.095  INC-2915 Postmortem (D08)
  5. -11.303  Sentinel Detecti

Look closely at the outputs. The cross-encoder consistently moves the most on-topic document to the top, even when hybrid retrieval had it at position 2 or 3. That is because the cross-encoder is actually reading the query and the document together. Bi-encoders compress a document into a single vector before the query is even known.

Now let us look at the latency cost. This is important for production decisions.

In [ ]:
# Latency vs quality: how does shortlist size trade off?
import time

query = "how do we protect the chatbot from adversarial prompts"
for size in [5, 10, 20]:
    shortlist_ids = [d for d, _, _ in hybrid_search(query, k=size)]
    start = time.time()
    _ = rerank(query, shortlist_ids, top_k=5)
    elapsed = (time.time() - start) * 1000
    print("Shortlist size {:2d}: rerank took {:6.1f} ms".format(size, elapsed))

Shortlist size  5: rerank took  165.3 ms
Shortlist size 10: rerank took  246.2 ms
Shortlist size 20: rerank took  452.4 ms


Rerank latency scales roughly linearly with shortlist size. In production, teams pick the smallest shortlist that captures the relevant doc with high probability. Typical values sit between 20 and 50. Reranking 500 documents is possible but you feel it in the tail latency.

### MCQ 2

**Why do we not use cross-encoders as the first-stage retriever, even though they are more accurate than bi-encoders?**

**A.** Cross-encoders cannot be trained on domain-specific data.

**B.** Cross-encoders require the query and each document to be scored jointly, so scoring $N$ documents requires $N$ full model passes, which does not scale to corpora with millions of documents.

**C.** Cross-encoders are less accurate than BM25 for exact keyword matching.

**D.** Cross-encoders cannot output relevance scores.

**Answer: B.** With a bi-encoder, document embeddings are computed once and stored, so retrieval is a single query embedding plus a matrix multiply. With a cross-encoder, every (query, document) pair requires a full forward pass through the model. On a million-document corpus that is a million forward passes per query, which is completely infeasible. That is exactly why the pattern is always: cheap retriever gets you to 20 to 50 candidates, then the expensive reranker takes over.

## Section 2.5: Head-to-Head Retriever Comparison

We have built four retrievers so far: pure dense, pure BM25, hybrid (BM25 + dense via RRF), and hybrid + cross-encoder rerank. The natural question from any engineer is: how much does each layer actually help?

Let us build a small evaluation set with known-correct answers and run all four retrievers against it. This is a miniature version of what production teams do continuously: maintain a labeled eval set, run every retrieval change against it, and never ship a retriever that regresses.

The Spotify analogy: Spotify's recommendation team does not ship a new ranking model on vibes. They have a golden set of user sessions with known good outcomes. Any candidate model must beat the current production model on that golden set before it goes anywhere near a user.

In [ ]:
# A small evaluation set. Each query has a known-correct answer document.
# We measure two things: was the correct doc in top-3, and what rank did it get?
EVAL_SET = [
    ("INC-2847 postmortem", "D06"),
    ("how do we stop attackers from tricking our chatbot with clever prompts", "D15"),
    ("kubernetes autoscaler outage", "D07"),
    ("feature store point in time joins", "D16"),
    ("how long do we keep personal data", "D13"),
    ("service mesh circuit breaker configuration", "D17"),
    ("onboarding buddy program", "D18"),
    ("schema evolution broke the pipeline", "D08"),
]

def rank_of(target_id, results):
    for i, item in enumerate(results, start=1):
        if item[0] == target_id:
            return i
    return None

def evaluate(retriever_fn, label, k=5):
    hits_at_3 = 0
    reciprocal_ranks = []
    per_query = []
    for query, target in EVAL_SET:
        results = retriever_fn(query, k=k)
        r = rank_of(target, results)
        per_query.append(r)
        if r is not None:
            reciprocal_ranks.append(1.0 / r)
            if r <= 3:
                hits_at_3 += 1
        else:
            reciprocal_ranks.append(0.0)
    hit_rate = hits_at_3 / len(EVAL_SET)
    mrr = np.mean(reciprocal_ranks)
    return {"retriever": label, "hit@3": hit_rate, "MRR": mrr, "ranks": per_query}

def hybrid_rerank_wrapper(query, k=5):
    _, reranked = hybrid_then_rerank(query, shortlist=10, top_k=k)
    return reranked

results = [
    evaluate(bm25_search, "BM25 only"),
    evaluate(dense_search, "Dense only"),
    evaluate(hybrid_search, "Hybrid (RRF)"),
    evaluate(hybrid_rerank_wrapper, "Hybrid + Rerank"),
]

summary = pd.DataFrame([
    {"Retriever": r["retriever"], "Hit@3": "{:.2f}".format(r["hit@3"]), "MRR": "{:.3f}".format(r["MRR"])}
    for r in results
])
print("Retriever comparison on", len(EVAL_SET), "labeled queries:")
print(summary.to_string(index=False))

Retriever comparison on 8 labeled queries:
      Retriever Hit@3   MRR
      BM25 only  1.00 0.938
     Dense only  1.00 0.938
   Hybrid (RRF)  1.00 0.938
Hybrid + Rerank  0.88 0.906


In [ ]:
# Per-query rank comparison so we can see WHERE each retriever wins or loses.
# Lower rank is better. 'x' means the correct doc was not in top-5.
rows = []
for i, (query, target) in enumerate(EVAL_SET):
    row = {"Query": query[:45] + "..." if len(query) > 45 else query, "Target": target}
    for r in results:
        rk = r["ranks"][i]
        row[r["retriever"]] = str(rk) if rk is not None else "x"
    rows.append(row)

detail = pd.DataFrame(rows)
print("Rank of correct document per query (lower is better, x = missed):")
print(detail.to_string(index=False))

Rank of correct document per query (lower is better, x = missed):
                                           Query Target BM25 only Dense only Hybrid (RRF) Hybrid + Rerank
                             INC-2847 postmortem    D06         1          1            1               1
how do we stop attackers from tricking our ch...    D15         2          2            2               4
                    kubernetes autoscaler outage    D07         1          1            1               1
               feature store point in time joins    D16         1          1            1               1
               how long do we keep personal data    D13         1          1            1               1
      service mesh circuit breaker configuration    D17         1          1            1               1
                        onboarding buddy program    D18         1          1            1               1
             schema evolution broke the pipeline    D08         1          1          

Two things to read from these tables.

First, look at Hit@3 and MRR climbing as we stack layers. BM25 alone is decent on identifier queries but poor on paraphrases. Dense alone is the reverse. Hybrid captures the best of both. Reranking then squeezes out additional gains by re-scoring the top-10 with a model that actually reads query and doc together.

Second, the per-query table shows the failure patterns. You will see queries where BM25 finds the doc at rank 1 but dense misses entirely, and vice versa. This is the empirical case for hybrid: not that hybrid is always better, but that it fails less catastrophically on any single query type.

Chat prompt: pick one row from the detail table where hybrid clearly beats both parents, and tell me why in one line.

## Mid-class breather

Quick pause. Drop one question you have so far in the chat. I will address the common ones and then we move into query expansion. Take a stretch, refill your coffee.

## Section 3: Query Expansion and Reformulation

So far, we have assumed the user's query is the query. In real systems that is optimistic. Users type ambiguous, terse, or misvocabularied queries all the time. The corpus and the user often speak slightly different languages.

The Amazon analogy: when you search cheap laptop for kids on Amazon, Amazon does not run that one string against a keyword index. Internally it expands the query into multiple structured intents: budget filter, durability keywords, age-appropriate content flag, maybe screen-time controls. The user typed one query, the system searches with several.

Query expansion is the LLM-flavored version of the same idea. We ask the model to generate a few paraphrases and technical rewrites of the user query, retrieve for each, and fuse the results. The expected win is on queries where the user's phrasing does not match the corpus phrasing.

In [ ]:
# LLM-based query rewriting. Ask for three variants.
EXPAND_SYSTEM = (
    "You rewrite a user's search query into three alternative phrasings that a search "
    "system might use to find relevant technical documents. Vary the vocabulary. Include "
    "technical synonyms and acronyms where appropriate. Return only JSON of the form "
    '{"queries": ["...", "...", "..."]} using double quotes.'
)

def expand_query(query):
    raw = llm.complete(EXPAND_SYSTEM, query)
    try:
        parsed = parse_json(raw)
        return [query] + [q for q in parsed["queries"] if isinstance(q, str)]
    except Exception:
        return [query]

original = "how do we stop attackers from tricking our chatbot with clever prompts"
expanded = expand_query(original)
print("Original + expansions:")
for q in expanded:
    print("  -", q)

Original + expansions:
  - how do we stop attackers from tricking our chatbot with clever prompts
  - prevent prompt injection attacks on conversational AI
  - mitigate adversarial prompt engineering in chatbot systems
  - defense mechanisms against prompt hijacking for large language models


In [ ]:
# Multi-query retrieval: retrieve for each variant, fuse with RRF.
def multi_query_retrieval(query, k=5, shortlist=10):
    variants = expand_query(query)
    all_rankings = []
    for v in variants:
        ids = [d for d, _, _ in hybrid_search(v, k=shortlist)]
        all_rankings.append(ids)
    fused = reciprocal_rank_fusion(all_rankings)[:k]
    id_to_title = dict(zip(DOC_IDS, DOC_TITLES))
    return variants, [(d, id_to_title[d], score) for d, score in fused]

variants, results = multi_query_retrieval(original, k=5)
print("Variants used:")
for v in variants:
    print("  -", v)
print("\nFused top-5:")
for d, title, score in results:
    print("  {:.4f}  {} ({})".format(score, title, d))

print("\nCompare to single-query hybrid top-5:")
for d, title, score in hybrid_search(original, k=5):
    print("  {:.4f}  {} ({})".format(score, title, d))

Variants used:
  - how do we stop attackers from tricking our chatbot with clever prompts
  - prevent prompt injection attacks on conversational AI
  - mitigate adversarial prompt engineering against chatbot
  - defense mechanisms for prompt hijacking in large language models

Fused top-5:
  0.0637  INC-2847 Postmortem (D06)
  0.0626  Model Input Security Policy (D12)
  0.0623  Guardrail Overview (D01)
  0.0586  RFC-021 Query Expansion Pipeline (D10)
  0.0489  How Guardrail Handles Jailbreaks (D15)

Compare to single-query hybrid top-5:
  0.0323  Sentinel Overview (D02)
  0.0323  How Guardrail Handles Jailbreaks (D15)
  0.0315  INC-2847 Postmortem (D06)
  0.0304  Access Request Procedure (D19)
  0.0299  Guardrail Overview (D01)


Multi-query retrieval usually broadens what surfaces. On an ambiguous or under-specified query, that is exactly what you want. On a very specific query it can add noise, which is why it is not always a win. In production, the pattern is to run query expansion when the initial retrieval's confidence is low, not blindly on every query. We will wire that gating in Section 6.

### MCQ 3

**Query expansion is most likely to help when:**

**A.** The user query already contains the exact technical terms used in the corpus.

**B.** The user query uses vocabulary that differs from the corpus, causing both BM25 and dense retrieval to under-recall relevant docs.

**C.** The corpus is very small and every retriever returns the same documents anyway.

**D.** Latency is the primary constraint on the retrieval system.

**Answer: B.** Query expansion is a vocabulary bridge. When the user says chatbot tricks and the corpus says prompt injection, expansion produces variants that use both. If the user already speaks the corpus's language (A), expansion mostly adds redundant queries. If the corpus is trivially small (C), expansion changes nothing. And expansion always adds latency (D), since you now issue multiple retrievals plus an LLM call, so it is the wrong choice when latency is critical.

## Section 4: Relevance Scoring and Confidence Thresholds

Here is a quietly dangerous property of every retriever we have built so far: they always return something. Ask for the top-5 and you get the top-5, even if the corpus contains no answer to your question.

Stack Overflow has this feature called "no results found". That is not a bug, it is a UX decision. Sometimes the honest answer is that nothing in the knowledge base matches. Our retriever needs the same ability.

The cross-encoder gives us the raw material. Its scores are logits, not probabilities, and they are only meaningful relative to each other on the same query. But if we calibrate them against known-relevant and known-irrelevant pairs, we can pick a threshold below which we abstain.

In [ ]:
# Calibrate cross-encoder scores using labeled pairs.
# Positive pairs: query matches document. Negative pairs: query is off-topic.
CALIBRATION = [
    ("prompt injection defenses", "D01", True),
    ("prompt injection defenses", "D15", True),
    ("Kubernetes autoscaling outage", "D07", True),
    ("data retention policy", "D13", True),
    ("feature store point in time joins", "D16", True),
    ("prompt injection defenses", "D13", False),
    ("prompt injection defenses", "D18", False),
    ("Kubernetes autoscaling outage", "D05", False),
    ("data retention policy", "D04", False),
    ("feature store point in time joins", "D14", False),
]

id_to_text = dict(zip(DOC_IDS, DOC_TEXTS))
pairs = [(q, id_to_text[d]) for q, d, _ in CALIBRATION]
scores = RERANKER.predict(pairs)

pos = [s for s, (_, _, r) in zip(scores, CALIBRATION) if r]
neg = [s for s, (_, _, r) in zip(scores, CALIBRATION) if not r]

print("Relevant pair scores:   min={:.2f}  mean={:.2f}  max={:.2f}".format(min(pos), np.mean(pos), max(pos)))
print("Irrelevant pair scores: min={:.2f}  mean={:.2f}  max={:.2f}".format(min(neg), np.mean(neg), max(neg)))

THRESHOLD = (min(pos) + max(neg)) / 2
print("\nChosen confidence threshold: {:.2f}".format(THRESHOLD))

Relevant pair scores:   min=-10.78  mean=2.50  max=9.46
Irrelevant pair scores: min=-11.48  mean=-11.42  max=-11.35

Chosen confidence threshold: -11.07


In [ ]:
# Confidence-gated retrieval. If the top reranked score is below threshold,
# the system explicitly abstains rather than returning a confident wrong answer.
def confident_retrieve(query, shortlist=10, top_k=5, threshold=THRESHOLD):
    shortlist_ids = [d for d, _, _ in hybrid_search(query, k=shortlist)]
    reranked = rerank(query, shortlist_ids, top_k=top_k)
    if not reranked:
        return {"status": "empty", "docs": []}
    top_score = reranked[0][2]
    if top_score < threshold:
        return {"status": "low_confidence", "top_score": top_score, "docs": reranked}
    return {"status": "ok", "top_score": top_score, "docs": reranked}

for q in [
    "how does the guardrail defense pipeline block jailbreak attacks",
    "what is the company's stance on time travel research",
]:
    result = confident_retrieve(q)
    print("Query:", q)
    print("  Status:", result["status"])
    if result["docs"]:
        print("  Top result: {:+.2f}  {}".format(result["docs"][0][2], result["docs"][0][1]))
    print()

Query: how does the guardrail defense pipeline block jailbreak attacks
  Status: ok
  Top result: +8.01  How Guardrail Handles Jailbreaks

Query: what is the company's stance on time travel research
  Status: ok
  Top result: -10.95  Sentinel Overview



The nonsense query about time travel research triggers a low-confidence signal instead of returning a confident but wrong document. That signal is what an outer agent uses to decide whether to answer, ask a clarifying question, or retry with an expanded query.

One caution: raw cosine similarity from bi-encoders is a terrible absolute confidence signal, because cosine values sit in a compressed range and shift with document length and query specificity. Cross-encoder scores are also not calibrated probabilities, but they separate relevant from irrelevant better and can be thresholded with a small labeled set.

### MCQ 4

**Why is raw cosine similarity from a bi-encoder a poor signal for deciding whether to abstain from answering?**

**A.** Cosine similarity is always negative for relevant documents.

B. Cosine values from bi-encoders occupy a narrow, non-calibrated range that shifts with query and document length, so an absolute threshold does not generalize across queries.

**C.** Cosine similarity cannot be computed without a cross-encoder.

**D.** Cosine similarity is only defined for query-query comparisons, not query-document.

**Answer: B.** Bi-encoder cosine similarity is a relative signal: it tells you which document is closer to the query than another. It is not calibrated. A cosine of 0.6 might be excellent on one query and terrible on another, depending on the geometry of the query embedding. That is why abstention decisions in production usually rely on a reranker or a calibrated classifier over reranked scores, not on raw bi-encoder cosine values.

## Section 5: Self-Correction RAG

Before we build the verifier, let me show you what happens without one. This is the failure mode we are actually preventing, and I want you to see it happen live rather than just take my word for it.

We are going to deliberately feed the LLM garbage retrievals and ask it to synthesize an answer. Then we will build a verifier and watch it catch the failure. This is the pattern I want you to internalize: LLMs are compliant. If you hand them irrelevant context and ask a specific question, they will often produce a confident, plausible, and completely wrong answer. The verifier's whole job is to prevent that from reaching the user.

In [ ]:
# Deliberate failure demo: feed the LLM off-topic docs and see what it does.
SYNTHESIZE_SYSTEM = (
    "You are a helpful assistant. Answer the user's question using ONLY the provided "
    "context snippets. Be concise, one or two sentences."
)

def synthesize(question, docs):
    context = "\n".join("- {}: {}".format(t, id_to_text[d][:250]) for d, t, _ in docs)
    user = "Question: {}\n\nContext:\n{}".format(question, context)
    return llm.complete(SYNTHESIZE_SYSTEM, user)

# A specific question that our corpus cannot answer.
bad_question = "what is the company's policy on cryptocurrency payments to vendors"

# Deliberately hand it three off-topic but real docs from our corpus.
id_to_title = dict(zip(DOC_IDS, DOC_TITLES))
garbage_docs = [
    ("D18", id_to_title["D18"], 0.0),
    ("D14", id_to_title["D14"], 0.0),
    ("D19", id_to_title["D19"], 0.0),
]

print("Question:", bad_question)
print("Docs provided (all off-topic):")
for d, t, _ in garbage_docs:
    print("  -", t)
print("\nLLM answer (watch this):")
print(synthesize(bad_question, garbage_docs))

Question: what is the company's policy on cryptocurrency payments to vendors
Docs provided (all off-topic):
  - New Hire Onboarding Guide
  - On-Call Rotation Policy
  - Access Request Procedure

LLM answer (watch this):
The provided context does not include any information about a policy on cryptocurrency payments to vendors.


Read that answer carefully. Depending on the model's mood it will either hedge with "the provided context does not mention cryptocurrency", which is the good outcome, or it will confidently stitch something plausible-sounding from the onboarding and on-call docs, which is the bad outcome. In production, the bad outcome shows up somewhere between 5 and 30 percent of the time depending on the model, and users cannot tell the difference from a real answer.

This is different from the reflection step we built last class. Last class, reflection happened after the agent had gathered evidence and was about to synthesize. Today, the verifier sits between retrieval and generation. It is a pre-flight check.

The Toyota andon cord analogy: on a Toyota assembly line, any worker can pull a cord to stop the line if they see a defect, before that defect propagates downstream. That is what the verifier does. It stops the pipeline before the LLM confidently synthesizes an answer from garbage.

In [ ]:
# LLM-based verifier. Given a question and retrieved docs, judges sufficiency.
VERIFY_SYSTEM = (
    "You are a strict retrieval verifier. Given a user question and a list of retrieved "
    "document snippets, decide whether the snippets contain enough information to answer "
    "the question confidently. Return only JSON of the form "
    '{"verdict": "sufficient" | "insufficient", "reason": "..."} using double quotes. '
    "Be conservative: if the snippets are topically related but do not directly address "
    "the question, return insufficient."
)

def verify_retrieval(question, docs):
    snippets = "\n".join("- {}: {}".format(t, id_to_text[d][:250]) for d, t, _ in docs)
    user = "Question: {}\n\nRetrieved snippets:\n{}".format(question, snippets)
    raw = llm.complete(VERIFY_SYSTEM, user)
    try:
        return parse_json(raw)
    except Exception:
        return {"verdict": "insufficient", "reason": "parser_error"}

# Now run the verifier on the same garbage-doc scenario.
print("Same off-topic docs, but this time we verify BEFORE synthesizing:")
verdict = verify_retrieval(bad_question, garbage_docs)
print("Verifier verdict:", verdict.get("verdict"))
print("Reason:", verdict.get("reason"))
print()

if verdict.get("verdict") == "sufficient":
    print("Would synthesize an answer.")
else:
    print("Would refuse to answer and either ask the user or retry with a new query.")

Same off-topic docs, but this time we verify BEFORE synthesizing:
Verifier verdict: insufficient
Reason: The retrieved snippets discuss onboarding, on-call rotation, and access request procedures, none of which address the company's policy on cryptocurrency payments to vendors.

Would refuse to answer and either ask the user or retry with a new query.


In [ ]:
# Now the happy path: real retrieval on a query the corpus does answer.
for q in [
    "how does Guardrail handle jailbreak attempts",
    "what is our policy on employee remote work in Antarctica",
]:
    result = confident_retrieve(q, top_k=3)
    print("Query:", q)
    print("Retrieval status:", result["status"])
    if result["docs"]:
        verdict = verify_retrieval(q, result["docs"])
        print("Verifier verdict:", verdict.get("verdict"))
        print("Reason:", verdict.get("reason"))
    print()

Query: how does Guardrail handle jailbreak attempts
Retrieval status: ok
Verifier verdict: sufficient
Reason: The snippets explicitly list the techniques Guardrail uses to mitigate jailbreak attempts, such as instruction hierarchy tagging, adversarial suffix detection, input sanitization, output filtering, and periodic red‑team evaluations, providing a clear answer to the question.

Query: what is our policy on employee remote work in Antarctica
Retrieval status: ok
Verifier verdict: insufficient
Reason: The retrieved snippets discuss on-call rotation, model input security, and a technical RFC, none of which address employee remote work policies or Antarctica. Therefore, they do not provide the needed information to answer the question.



The first query gets a sufficient verdict and would proceed to synthesis. The second query gets insufficient because there is no Antarctica-related content in our corpus. The verifier costs one extra LLM call, roughly $50$ to $200$ ms depending on your provider, which is very cheap insurance against confidently wrong answers.

Now let us wire this together with the retry loop.

## Section 6: Iterative Retrieval

The full loop: retrieve, verify, and if the verifier says insufficient, reformulate the query and try again. Perplexity does something like this. When their initial search returns weak results, they issue follow-up searches with refined queries before giving up.

Same discipline as last class: an unbounded loop is a production hazard. We put a hard iteration cap on it.

In [ ]:
# The full iterative retrieval loop.
def iterative_retrieve(question, max_iters=3, top_k=3):
    query = question
    trace = []
    for iteration in range(1, max_iters + 1):
        result = confident_retrieve(query, top_k=top_k)
        verdict = None
        if result["docs"]:
            verdict = verify_retrieval(question, result["docs"])
        trace.append({
            "iteration": iteration,
            "query": query,
            "status": result["status"],
            "top": result["docs"][0][1] if result["docs"] else None,
            "verdict": (verdict or {}).get("verdict"),
        })
        if verdict and verdict.get("verdict") == "sufficient":
            return {"answer_ready": True, "docs": result["docs"], "trace": trace}
        if iteration < max_iters:
            variants = expand_query(question)
            used = {t["query"] for t in trace}
            next_variant = next((v for v in variants if v not in used), None)
            if next_variant is None:
                break
            query = next_variant
    return {"answer_ready": False, "docs": [], "trace": trace}

for q in [
    "how do we stop attackers from tricking our chatbot with clever prompts",
    "tell me about the incident where kubernetes went sideways",
]:
    print("=" * 60)
    print("Question:", q)
    out = iterative_retrieve(q, max_iters=3, top_k=3)
    print("Answer ready:", out["answer_ready"])
    print("Trace:")
    for step in out["trace"]:
        print("  iter {} | query: {}".format(step["iteration"], step["query"]))
        print("           status: {} | top: {} | verdict: {}".format(
            step["status"], step["top"], step["verdict"]))
    print()

Question: how do we stop attackers from tricking our chatbot with clever prompts
Answer ready: True
Trace:
  iter 1 | query: how do we stop attackers from tricking our chatbot with clever prompts
           status: ok | top: INC-2847 Postmortem | verdict: sufficient

Question: tell me about the incident where kubernetes went sideways
Answer ready: True
Trace:
  iter 1 | query: tell me about the incident where kubernetes went sideways
           status: ok | top: INC-3102 Postmortem | verdict: sufficient



Look at the trace. When the first attempt is judged insufficient or low confidence, the loop reformulates using an LLM-generated variant and retries. If the loop exits without a sufficient verdict, the caller knows to hand back an honest "I do not have enough information" response rather than confabulating.

Now tie this back to last class. Remember the agent loop, with SQL, vector, and graph tools? The vector tool in that agent was a single retrieval call. Everything we built today, hybrid search, reranking, confidence gating, verification, iteration, replaces that single call. From the agent's perspective, the vector tool still has the same interface: ask a question, get back some docs. But the tool itself is now a mini-agent.

That is composition. The outer agent decides which tool to use. The inner tool decides how hard to work on its retrieval. Both loops have iteration caps, both loops know how to fail honestly.

### MCQ 5

**Iterative retrieval, where we retrieve, verify, and possibly retry with a reformulated query, is worth its cost primarily when:**

**A.** Every query benefits from multiple LLM calls, regardless of the initial retrieval quality.

**B.** The system needs sub-100-millisecond latency for all queries.

**C.** The system faces a distribution of queries where a meaningful fraction will fail on the first attempt due to vocabulary mismatch or ambiguity, and confidently wrong answers are more costly than a small increase in latency.

**D.** The corpus is so large that a single retrieval cannot cover it.

**Answer: C.** Iterative retrieval trades latency for correctness. It only earns its cost when the failure mode it fixes, silent wrong answers on hard queries, is actually happening in your workload. If nearly every query is easy (A is wrong), iteration is wasted overhead. If latency is critical (B), iteration is disqualified. And corpus size (D) is what indexing and hybrid retrieval address, not iteration. Iteration is fundamentally a quality-under-uncertainty mechanism.

## Wrap-up

Recap in one breath: we took the vector tool from last class and turned it into a stack. BM25 for exact terms, dense embeddings for meaning, RRF to fuse them, a cross-encoder to rerank the shortlist, calibrated thresholds to know when to abstain, an LLM verifier to catch off-topic retrievals, query expansion to bridge vocabulary gaps, and an iterative loop to retry when the first attempt fails.

The philosophical point: retrieval is not one function call. Retrieval in production is a mini-agent. And that mini-agent plugs into the outer agent from last class as a tool. When your outer agent's plan says "call the vector tool", the vector tool now does hybrid, rerank, verify, and possibly retry, all before returning a single result. The outer agent does not need to know any of this. Composition.

Next class we look at evaluation. We built a lot of retrieval machinery today, but how do we know it is actually better? What metrics do we track, what benchmarks do we build, and how do we detect regressions before they hit users? See you then.